# 08 — Scenario Delta Maps (Publication Figure)

**Purpose:** Primary results figure for the paper.  For each policy scenario
relative to the baseline, maps three metrics across all 67 BA zones:

| Row | Metric | Unit | Scale |
|-----|--------|------|-------|
| 1 | ΔLMP | \$/MWh | RdBu_r diverging |
| 2 | ΔCO₂ Emissions | Mt CO₂ | RdBu diverging |
| 3 | ΔClean-energy share | percentage points | RdYlGn diverging |

Columns = policy scenarios vs baseline.  Diverging scales are centred at zero.
Each row shares a single colorbar so panels are directly comparable.

**Note:** The `ces_achievable` (50% CES) constraint is non-binding given the
current generation fleet — all deltas for that scenario are identically zero.
This is physically meaningful and the maps are retained to document
non-bindingness.  The 80% CES target was infeasible.

**Inputs:**
- `data/processed/e4st_results/{scenario}/lmp.parquet`
- `data/processed/e4st_results/{scenario}/dispatch.parquet`
- `data/processed/ba_territories.geojson`
- `data/processed/network_metadata.json`

**Outputs:**
- `data/processed/figures/scenario_compare.png` (300 DPI)

In [ ]:
import sys, json, textwrap
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm

FIGURES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Publication rcParams
mpl.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':          9,
    'axes.linewidth':     0.6,
    'xtick.major.width':  0.5,
    'ytick.major.width':  0.5,
    'figure.dpi':        120,   # screen preview; saved at 300
})

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RESULTS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'e4st_results'
BA_GEO_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ba_territories.geojson'
META_PATH   = PROJECT_ROOT / 'data' / 'processed' / 'network_metadata.json'

with open(META_PATH) as f:
    meta = json.load(f)

SCENARIOS = [s['name'] for s in meta['scenarios_completed'] if s['status'] == 'OPTIMAL']
POLICY_SCENARIOS = [s for s in SCENARIOS if s != 'baseline']

# LaTeX-safe labels for axes and caption
SCENARIO_LABELS = {
    'carbon_tax_50':  'Carbon Tax $50/tCO$_2$',
    'ces_achievable': 'CES 50\\% (max feasible)',
}

CLEAN_FUELS = {'wind', 'solar', 'hydro', 'nuclear', 'geothermal', 'biomass'}

print('Policy scenarios vs baseline:', POLICY_SCENARIOS)

In [ ]:
# ── Load results for all scenarios + geometries ────────────────────────────────
lmp_data      = {}
dispatch_data = {}
for sc in SCENARIOS:
    lmp_data[sc]      = pd.read_parquet(RESULTS_DIR / sc / 'lmp.parquet')
    dispatch_data[sc] = pd.read_parquet(RESULTS_DIR / sc / 'dispatch.parquet')

# Reproject to Conus Albers (EPSG:5070) for publication maps
ba_geo = (
    gpd.read_file(BA_GEO_PATH)[['ba_code', 'ba_name', 'geometry']]
    .to_crs(epsg=5070)
)
print(f'BA territories: {len(ba_geo)} polygons, CRS EPSG:{ba_geo.crs.to_epsg()}')

In [ ]:
# ── Compute per-BA delta metrics ───────────────────────────────────────────────
def clean_share(disp_df):
    """Fraction of each BA's annual generation from clean fuels."""
    tot = disp_df.groupby('ba')['dispatch_mwh'].sum()
    cln = (
        disp_df[disp_df['genfuel'].isin(CLEAN_FUELS)]
        .groupby('ba')['dispatch_mwh'].sum()
    )
    return (cln / tot).rename('clean_share').fillna(0)

base_lmp   = lmp_data['baseline'].set_index('ba')['lmp_mwh']
base_co2   = dispatch_data['baseline'].groupby('ba')['co2_emitted_tons'].sum()
base_clean = clean_share(dispatch_data['baseline'])

deltas = {}
for sc in POLICY_SCENARIOS:
    dlmp   = (lmp_data[sc].set_index('ba')['lmp_mwh'] - base_lmp).rename('delta_lmp')
    dco2   = (
        dispatch_data[sc].groupby('ba')['co2_emitted_tons'].sum() - base_co2
    ).rename('delta_co2_tons')
    dclean = (clean_share(dispatch_data[sc]) - base_clean).rename('delta_clean_pp') * 100
    deltas[sc] = pd.concat([dlmp, dco2, dclean], axis=1)
    print(f'{sc}: mean ΔLMP={dlmp.mean():+.3f} $/MWh, '
          f'ΔCO2={dco2.sum()/1e6:+.3f} Mt, '
          f'ΔClean={dclean.mean():+.3f} pp')

In [ ]:
# ── Merge deltas onto BA geometries ───────────────────────────────────────────
geo_deltas = {}
for sc in POLICY_SCENARIOS:
    gdf = ba_geo.merge(
        deltas[sc].reset_index(),
        left_on='ba_code', right_on='ba', how='left'
    )
    geo_deltas[sc] = gdf
    print(f'{sc}: {gdf["delta_lmp"].notna().sum()} zones with data')

In [ ]:
# ── Symmetric diverging color-scale limits (98th pct of |delta|) ──────────────
def symmax(series_list, pct=98):
    combined = pd.concat(series_list).dropna()
    return float(np.percentile(combined.abs(), pct)) or 1e-9

vmax_lmp   = symmax([deltas[sc]['delta_lmp']                      for sc in POLICY_SCENARIOS])
vmax_co2   = symmax([deltas[sc]['delta_co2_tons'] / 1e6           for sc in POLICY_SCENARIOS])
vmax_clean = symmax([deltas[sc]['delta_clean_pp']                  for sc in POLICY_SCENARIOS])

# If a scenario has all-zero deltas (non-binding), avoid vmax=0
vmax_lmp   = max(vmax_lmp,   1e-3)
vmax_co2   = max(vmax_co2,   1e-6)
vmax_clean = max(vmax_clean, 1e-6)

METRICS = [
    dict(col='delta_lmp',       scale=1.0,   label='\u0394LMP ($/MWh)',
         cmap='RdBu_r', vmax=vmax_lmp,   row_letter='\u0394LMP'),
    dict(col='delta_co2_tons',  scale=1e-6,  label='\u0394CO\u2082 (Mt)',
         cmap='RdBu',   vmax=vmax_co2,   row_letter='\u0394CO\u2082'),
    dict(col='delta_clean_pp',  scale=1.0,   label='\u0394Clean Share (pp)',
         cmap='RdYlGn', vmax=vmax_clean, row_letter='\u0394Clean'),
]

for m in METRICS:
    print(f'{m["label"]:25s}: \u00b1{m["vmax"]:.4f}')

In [ ]:
# ── Map-panel helper ───────────────────────────────────────────────────────────
def draw_map(ax, gdf, col, scale, cmap_name, vmax, title, annotate_nonbinding=False):
    """Draw one choropleth panel onto ax."""
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    gdf_na  = gdf[gdf[col].isna()]
    gdf_val = gdf[gdf[col].notna()].copy()
    gdf_val['_v'] = gdf_val[col] * scale

    if not gdf_na.empty:
        gdf_na.plot(ax=ax, color='#e0e0e0', linewidth=0.15, edgecolor='#cccccc')

    gdf_val.plot(
        ax=ax, column='_v', cmap=cmap_name, norm=norm,
        linewidth=0.25, edgecolor='#888888',
    )

    ax.set_title(title, fontsize=9, pad=4)
    ax.axis('off')

    if annotate_nonbinding:
        ax.text(
            0.5, 0.04,
            'Constraint non-binding \u2014 all \u0394 = 0',
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=7.5, color='#444444', style='italic',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#aaaaaa', lw=0.6),
        )

    return norm

In [ ]:
# ── Build publication figure ───────────────────────────────────────────────────
N_ROWS = len(METRICS)
N_COLS = len(POLICY_SCENARIOS)

# Extra narrow column on right holds shared colorbars
fig, axes = plt.subplots(
    N_ROWS, N_COLS + 1,
    figsize=(6.5 * N_COLS + 1.0, 4.2 * N_ROWS),
    gridspec_kw={'width_ratios': [6.5] * N_COLS + [0.35]},
)
fig.subplots_adjust(left=0.04, right=0.97, top=0.92, bottom=0.03,
                    hspace=0.08, wspace=0.04)

panel_letters = list('abcdefghij')
letter_idx = 0

for row_i, metric in enumerate(METRICS):
    col_val  = metric['col']
    scale    = metric['scale']
    vmax     = metric['vmax']
    cmap_nm  = metric['cmap']
    cb_label = metric['label']
    norm_obj = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    for col_i, sc in enumerate(POLICY_SCENARIOS):
        ax     = axes[row_i, col_i]
        gdf    = geo_deltas[sc]
        letter = panel_letters[letter_idx]; letter_idx += 1
        title  = f'({letter})'

        # Is this scenario's delta identically zero?
        nonbinding = (deltas[sc][col_val].abs().max() < 1e-6)

        draw_map(ax, gdf, col_val, scale, cmap_nm, vmax, title, nonbinding)

        # Row label on left edge of first column
        if col_i == 0:
            ax.text(
                -0.03, 0.5, metric['row_letter'],
                transform=ax.transAxes, va='center', ha='right',
                fontsize=9, fontweight='bold', rotation=90,
            )

    # Shared colorbar
    cb_ax = axes[row_i, N_COLS]
    sm = cm.ScalarMappable(norm=norm_obj, cmap=plt.get_cmap(cmap_nm))
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cb_ax, orientation='vertical')
    cb.set_label(cb_label, fontsize=8)
    cb.ax.tick_params(labelsize=7)
    cb.ax.axhline(0.5, color='black', linewidth=0.8, linestyle='--')  # zero line

# Column headers
col_header_labels = {
    'carbon_tax_50':  'Carbon Tax $50/tCO\u2082  vs  Baseline',
    'ces_achievable': 'CES 50% (max feasible)  vs  Baseline',
}
for col_i, sc in enumerate(POLICY_SCENARIOS):
    axes[0, col_i].set_title(
        col_header_labels.get(sc, sc),
        fontsize=10, fontweight='bold', pad=10, loc='center',
    )

fig.suptitle(
    'E4ST Zonal Copper-Plate Model \u2014 Scenario Delta Maps',
    fontsize=12, fontweight='bold', y=0.97,
)

out = FIGURES_DIR / 'scenario_compare.png'
fig.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved \u2192 {out}')
plt.show()

In [ ]:
# ── Summary statistics table (one per scenario) ────────────────────────────────
def delta_stats(sc):
    d = deltas[sc]
    rows = []
    for col, unit, scale in [
        ('delta_lmp',      '$/MWh',    1.0),
        ('delta_co2_tons', 'Mt CO\u2082', 1e-6),
        ('delta_clean_pp', 'pp',        1.0),
    ]:
        s = d[col].dropna() * scale
        rows.append({
            'Metric':  f'\u0394{col.replace("delta_","")} ({unit})',
            'Mean':    s.mean(),
            'Std':     s.std(),
            'Min':     s.min(),
            'Max':     s.max(),
            'Sum':     s.sum(),
        })
    return pd.DataFrame(rows).set_index('Metric')

for sc in POLICY_SCENARIOS:
    label = col_header_labels.get(sc, sc)
    print(f'\n\u2500 {label} \u2500')
    display(delta_stats(sc).style.format('{:+.4f}'))

In [ ]:
# ── LaTeX figure caption ───────────────────────────────────────────────────────
# Numbers are pulled directly from the computed deltas so the caption
# stays in sync if data changes.
ct  = deltas['carbon_tax_50']
ces = deltas['ces_achievable']

mean_dlmp_ct   = ct['delta_lmp'].mean()
sum_dco2_ct    = ct['delta_co2_tons'].sum() / 1e6
max_dlmp_ces   = ces['delta_lmp'].abs().max()

caption = textwrap.dedent(f"""\
    \\begin{{figure}}[htbp]
      \\centering
      \\includegraphics[width=\\textwidth]{{figures/scenario_compare}}
      \\caption{{%
        Delta maps comparing each policy scenario against the no-policy baseline
        in the E4ST zonal copper-plate model (67~balancing-authority zones,
        2024~capacity vintage).
        Rows show change in
        (i)~mean LMP (\$/MWh),
        (ii)~annual CO\\textsubscript{{2}} emissions (Mt), and
        (iii)~clean-energy dispatch share (percentage points).
        \\emph{{Left column: carbon tax.}}
        A \\$50/tCO\\textsubscript{{2}} carbon tax raises mean zonal prices by
        \\${mean_dlmp_ct:+.2f}\\,\$/MWh
        and reduces system-wide CO\\textsubscript{{2}} emissions by
        {abs(sum_dco2_ct):.1f}\\,Mt relative to baseline.
        \\emph{{Right column: clean-energy standard.}}
        The 50\\,\% CES (the maximum feasible fraction; the 80\\,\% target
        is infeasible with the current fleet) is non-binding---all
        deltas are zero to machine precision
        ($|\\Delta\\text{{LMP}}|_{{\\max}}={max_dlmp_ces:.2e}$\\,\$/MWh).
        Diverging colour scales are symmetric about zero and shared within each row;
        grey polygons indicate BA territories outside the 67-zone model domain.
      }}
      \\label{{fig:scenario_compare}}
    \\end{{figure}}"""
)

print(caption)